In [5]:

# LAB 06 - PREPARE MRI TEST DATASET
# Extract ZIP, fix Windows paths, verify 995 images

from pathlib import Path
from zipfile import ZipFile
import shutil

ZIP_PATH = Path("lab06_test_995.zip")
DATASET_ROOT = Path("test_data")

EXPECTED = {
    "glioma": 253,
    "meningioma": 303,
    "no_tumor": 139,
    "pituitary": 300
}

EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp"}
DATASET_ROOT.mkdir(exist_ok=True)

# 1. Fix previously extracted Windows-style paths
fixed = 0

for src in list(DATASET_ROOT.iterdir()):
    if not src.is_file() or "\\" not in src.name:
        continue

    parts = src.name.replace("\\", "/").split("/")

    if len(parts) < 2:
        continue

    label, filename = parts[-2], parts[-1]

    if label not in EXPECTED:
        continue
    if Path(filename).suffix.lower() not in EXTENSIONS:
        continue

    dest = DATASET_ROOT / label / filename
    dest.parent.mkdir(exist_ok=True)

    if dest.exists():
        raise FileExistsError(f"Duplicate image: {dest}")

    src.rename(dest)
    fixed += 1

# 2. Count existing images
def count_images(label):
    folder = DATASET_ROOT / label

    if not folder.is_dir():
        return 0

    return sum(
        1 for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    )

# 3. Extract missing images directly into correct folders
if any(count_images(c) != n for c, n in EXPECTED.items()):
    assert ZIP_PATH.is_file(), "ZIP file not found!"

    with ZipFile(ZIP_PATH, "r") as z:
        for member in z.infolist():
            parts = member.filename.replace("\\", "/").split("/")

            if len(parts) < 2:
                continue

            label, filename = parts[-2], parts[-1]

            if label not in EXPECTED:
                continue
            if Path(filename).suffix.lower() not in EXTENSIONS:
                continue

            dest = DATASET_ROOT / label / filename
            dest.parent.mkdir(exist_ok=True)

            if not dest.exists():
                with z.open(member) as source:
                    with open(dest, "wb") as output:
                        shutil.copyfileobj(source, output)

# 4. Verify all MRI images
print("=== LAB 06 - MRI TEST DATASET ===")
print("Dataset path:", DATASET_ROOT.resolve())
print("Windows paths fixed:", fixed)
print()

total = 0

for label, expected in EXPECTED.items():
    count = count_images(label)
    total += count

    print(f"{label}: {count} images")
    assert count == expected, f"Count mismatch: {label}"

print("\nTotal:", total)
assert total == 995, "Dataset count mismatch!"

print("\nPASS: All 995 MRI test images are ready!")


=== LAB 06 - MRI TEST DATASET ===
Dataset path: /home/xilinx/jupyter_notebooks/Lab_06/test_data
Windows paths fixed: 0

glioma: 253 images
meningioma: 303 images
no_tumor: 139 images
pituitary: 300 images

Total: 995

PASS: All 995 MRI test images are ready!


In [6]:

# LAB 06 - EVALUATION ON 995 MRI IMAGES
# Compare ARM-only and ARM + FPGA on PYNQ-Z2

import time
import csv
from pathlib import Path
import numpy as np
from PIL import Image
from pynq import Overlay, allocate
from tflite_runtime.interpreter import Interpreter

LABELS = ["glioma", "meningioma", "no_tumor", "pituitary"]
EXPECTED = [253, 303, 139, 300]
DATA = Path("test_data")
ASSETS = Path("../Lab_05")
EXT = {".jpg", ".jpeg", ".png", ".bmp"}

# 1. Collect labeled test images
samples = []
for i, name in enumerate(LABELS):
    paths = sorted(
        p for p in (DATA / name).rglob("*")
        if p.is_file() and p.suffix.lower() in EXT
    )
    assert len(paths) == EXPECTED[i], name
    samples.extend((p, i) for p in paths)

assert len(samples) == 995

for filename in [
    "conv1_accel.bit",
    "conv1_accel.hwh",
    "tiny_cnn_v5_int8.tflite",
    "tiny_cnn_v5_tail_int8.tflite"
]:
    assert (ASSETS / filename).is_file(), filename

# 2. Load FPGA and TFLite models
ol = Overlay(str((ASSETS / "conv1_accel.bit").resolve()))
dma, conv1 = ol.axi_dma_0, ol.conv1_hls_0

full = Interpreter(
    model_path=str(ASSETS / "tiny_cnn_v5_int8.tflite"),
    num_threads=1
)
tail = Interpreter(
    model_path=str(ASSETS / "tiny_cnn_v5_tail_int8.tflite"),
    num_threads=1
)
full.allocate_tensors()
tail.allocate_tensors()

fi = full.get_input_details()[0]["index"]
fo = full.get_output_details()[0]["index"]
ti = tail.get_input_details()[0]["index"]
to = tail.get_output_details()[0]["index"]

assert tuple(tail.get_input_details()[0]["shape"]) == (1,47,47,8)

ibuf = allocate(shape=(9216,), dtype=np.int8)
obuf = allocate(shape=(47,47,8), dtype=np.int8)

# 3. Image preprocessing
def prepare(path):
    with Image.open(path) as img:
        img = img.convert("L").resize((96,96), Image.BILINEAR)
        pixels = np.asarray(img, dtype=np.int16)
    return (pixels - 128).astype(np.int8).reshape(1,96,96,1)

# 4. ARM-only inference
def run_arm(x):
    t0 = time.perf_counter()
    full.set_tensor(fi, x)
    full.invoke()
    result = full.get_tensor(fo).copy()
    return result, (time.perf_counter()-t0)*1000

# 5. FPGA Conv1 followed by ARM tail
def run_hybrid(x):
    t0 = time.perf_counter()

    ibuf[:] = x.reshape(-1)
    ibuf.flush()

    dma.recvchannel.transfer(obuf)
    conv1.write(0x00, 1)
    dma.sendchannel.transfer(ibuf)

    deadline = time.monotonic() + 3.0
    while True:
        tx = dma.mmio.read(0x04)
        rx = dma.mmio.read(0x34)

        if (tx | rx) & 0x70:
            raise RuntimeError(f"DMA error: {hex(tx)}, {hex(rx)}")
        if (tx & 2) and (rx & 2):
            break
        if time.monotonic() >= deadline:
            raise TimeoutError(f"DMA timeout: {hex(tx)}, {hex(rx)}")

    obuf.invalidate()
    t1 = time.perf_counter()

    pool1 = np.asarray(obuf).reshape(1,47,47,8)
    tail.set_tensor(ti, pool1)
    tail.invoke()
    result = tail.get_tensor(to).copy()
    t2 = time.perf_counter()

    return result, (t1-t0)*1000, (t2-t1)*1000, (t2-t0)*1000

# 6. Warm-up
print("Warming up...")
for p, _ in samples[::200][:5]:
    x = prepare(p)
    a, _ = run_arm(x)
    b, _, _, _ = run_hybrid(x)
    assert np.array_equal(a, b), "Warm-up output mismatch"

# 7. Evaluate all 995 images
print("Evaluating 995 MRI images...")

records = []
cm_arm = np.zeros((4,4), dtype=np.int64)
cm_hybrid = np.zeros((4,4), dtype=np.int64)

for n, (path, label) in enumerate(samples, 1):
    x = prepare(path)

    # Alternate execution order to reduce timing bias
    if n % 2:
        a, arm_ms = run_arm(x)
        b, fpga_ms, tail_ms, hybrid_ms = run_hybrid(x)
    else:
        b, fpga_ms, tail_ms, hybrid_ms = run_hybrid(x)
        a, arm_ms = run_arm(x)

    arm_pred = int(np.argmax(a))
    hybrid_pred = int(np.argmax(b))
    match = bool(np.array_equal(a, b))

    cm_arm[label, arm_pred] += 1
    cm_hybrid[label, hybrid_pred] += 1

    records.append([
        str(path), LABELS[label],
        LABELS[arm_pred], LABELS[hybrid_pred],
        int(match), arm_ms, fpga_ms, tail_ms, hybrid_ms
    ])

    if n % 100 == 0 or n == 995:
        print(f"Completed {n}/995 images")

# 8. Save individual predictions
header = [
    "image", "true_label", "arm_prediction",
    "hybrid_prediction", "outputs_match",
    "arm_ms", "fpga_dma_ms", "tail_ms", "hybrid_ms"
]

with open("lab06_predictions.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(header)
    w.writerows(records)

# 9. Calculate metrics
def metrics(cm):
    tp = np.diag(cm).astype(float)
    precision = tp / np.maximum(cm.sum(axis=0), 1)
    recall = tp / np.maximum(cm.sum(axis=1), 1)
    f1 = 2*precision*recall / np.maximum(precision+recall, 1e-12)
    accuracy = tp.sum() / cm.sum()
    return accuracy, precision, recall, f1

def save_matrix(name, cm):
    with open(name, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["Actual / Predicted"] + LABELS)
        for label, row in zip(LABELS, cm):
            w.writerow([label] + row.tolist())

save_matrix("lab06_confusion_arm.csv", cm_arm)
save_matrix("lab06_confusion_hybrid.csv", cm_hybrid)

acc_a, prec_a, rec_a, f1_a = metrics(cm_arm)
acc_h, prec_h, rec_h, f1_h = metrics(cm_hybrid)

times = np.asarray([r[5:] for r in records], dtype=float)
arm_med, fpga_med, tail_med, hybrid_med = np.median(times, axis=0)

exact_matches = sum(r[4] for r in records)

lines = [
    "=== LAB 06 - 995 MRI TEST IMAGES ===",
    f"Total images: {len(records)}",
    f"Identical ARM/Hybrid outputs: {exact_matches}/995",
    f"ARM accuracy: {acc_a*100:.2f}%",
    f"Hybrid accuracy: {acc_h*100:.2f}%",
    f"ARM macro F1: {np.mean(f1_a):.4f}",
    f"Hybrid macro F1: {np.mean(f1_h):.4f}",
    "",
    "=== MEDIAN INFERENCE TIME ===",
    f"ARM only: {arm_med:.3f} ms",
    f"FPGA + DMA: {fpga_med:.3f} ms",
    f"ARM tail: {tail_med:.3f} ms",
    f"ARM + FPGA: {hybrid_med:.3f} ms",
    f"Speedup (ratio of medians): {arm_med/hybrid_med:.3f}x",
    f"Speedup (total inference time): "
    f"{times[:,0].sum()/times[:,3].sum():.3f}x",
    "",
    "=== CONFUSION MATRIX - ARM ===",
    str(cm_arm),
    "",
    "=== CONFUSION MATRIX - HYBRID ===",
    str(cm_hybrid),
    "",
    "=== PER-CLASS PRECISION / RECALL / F1 (HYBRID) ==="
]

for i, name in enumerate(LABELS):
    lines.append(
        f"{name}: P={prec_h[i]:.4f} "
        f"R={rec_h[i]:.4f} F1={f1_h[i]:.4f}"
    )

summary = "\n".join(lines)
print("\n" + summary)

Path("lab06_summary.txt").write_text(summary, encoding="utf-8")

ibuf.freebuffer()
obuf.freebuffer()

print("\nPASS: Evaluation completed.")
print("Saved CSV and summary files.")


Warming up...
Evaluating 995 MRI images...
Completed 100/995 images
Completed 200/995 images
Completed 300/995 images
Completed 400/995 images
Completed 500/995 images
Completed 600/995 images
Completed 700/995 images
Completed 800/995 images
Completed 900/995 images
Completed 995/995 images

=== LAB 06 - 995 MRI TEST IMAGES ===
Total images: 995
Identical ARM/Hybrid outputs: 995/995
ARM accuracy: 92.66%
Hybrid accuracy: 92.66%
ARM macro F1: 0.9194
Hybrid macro F1: 0.9194

=== MEDIAN INFERENCE TIME ===
ARM only: 23.644 ms
FPGA + DMA: 1.452 ms
ARM tail: 16.431 ms
ARM + FPGA: 17.855 ms
Speedup (ratio of medians): 1.324x
Speedup (total inference time): 1.324x

=== CONFUSION MATRIX - ARM ===
[[244   7   0   2]
 [ 17 264  12  10]
 [  3  13 117   6]
 [  1   2   0 297]]

=== CONFUSION MATRIX - HYBRID ===
[[244   7   0   2]
 [ 17 264  12  10]
 [  3  13 117   6]
 [  1   2   0 297]]

=== PER-CLASS PRECISION / RECALL / F1 (HYBRID) ===
glioma: P=0.9208 R=0.9644 F1=0.9421
meningioma: P=0.9231 R=0.8